# Autoscaling a SageMaker Endpoint

In [1]:
import boto3
import sagemaker
import pandas as pd

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

sm = boto3.Session().client(service_name="sagemaker", region_name=region)
autoscale = boto3.Session().client(service_name="application-autoscaling", region_name=region)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r tensorflow_endpoint_name

In [3]:
try:
    tensorflow_endpoint_name
    print("[OK]")
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] Please run the notebooks in the previous notebook before you continue.")
    print("+++++++++++++++++++++++++++++++")

[OK]


In [4]:
print(tensorflow_endpoint_name)

tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736


# Copy the Model to the Notebook

In [5]:
autoscale.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId="endpoint/" + tensorflow_endpoint_name + "/variant/AllTraffic",
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=1,
    MaxCapacity=2,
    RoleARN=role,
    SuspendedState={
        "DynamicScalingInSuspended": False,
        "DynamicScalingOutSuspended": False,
        "ScheduledScalingSuspended": False,
    },
)

{'ScalableTargetARN': 'arn:aws:application-autoscaling:us-east-1:891377026966:scalable-target/056m30432a2a2e114a4791d38434d29778f1',
 'ResponseMetadata': {'RequestId': '1bf7b598-8642-4b57-83f2-f1bf01534513',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '1bf7b598-8642-4b57-83f2-f1bf01534513',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '131',
   'date': 'Wed, 03 Jul 2024 02:02:17 GMT'},
  'RetryAttempts': 0}}

In [6]:
# check the target is available
autoscale.describe_scalable_targets(
    ServiceNamespace="sagemaker",
    MaxResults=100,
)

{'ScalableTargets': [{'ServiceNamespace': 'sagemaker',
   'ResourceId': 'endpoint/tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736/variant/AllTraffic',
   'ScalableDimension': 'sagemaker:variant:DesiredInstanceCount',
   'MinCapacity': 1,
   'MaxCapacity': 2,
   'RoleARN': 'arn:aws:iam::891377026966:role/aws-service-role/sagemaker.application-autoscaling.amazonaws.com/AWSServiceRoleForApplicationAutoScaling_SageMakerEndpoint',
   'CreationTime': datetime.datetime(2024, 7, 3, 2, 2, 17, 722000, tzinfo=tzlocal()),
   'SuspendedState': {'DynamicScalingInSuspended': False,
    'DynamicScalingOutSuspended': False,
    'ScheduledScalingSuspended': False},
   'ScalableTargetARN': 'arn:aws:application-autoscaling:us-east-1:891377026966:scalable-target/056m30432a2a2e114a4791d38434d29778f1'}],
 'ResponseMetadata': {'RequestId': '19063fb7-a3c3-435e-9019-d201427c3008',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '19063fb7-a3c3-435e-9019-d201427c3008',
   'content-type': 

In [7]:
autoscale.put_scaling_policy(
    PolicyName="bert-reviews-autoscale-policy",
    ServiceNamespace="sagemaker",
    ResourceId="endpoint/" + tensorflow_endpoint_name + "/variant/AllTraffic",
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 2.0,
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerVariantInvocationsPerInstance",
        },
        "ScaleOutCooldown": 60,
        "ScaleInCooldown": 300,
    },
)

{'PolicyARN': 'arn:aws:autoscaling:us-east-1:891377026966:scalingPolicy:30432a2a-2e11-4a47-91d3-8434d29778f1:resource/sagemaker/endpoint/tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736/variant/AllTraffic:policyName/bert-reviews-autoscale-policy',
 'Alarms': [{'AlarmName': 'TargetTracking-endpoint/tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736/variant/AllTraffic-AlarmHigh-e394e480-4118-411f-9b4a-d836ee94d669',
   'AlarmARN': 'arn:aws:cloudwatch:us-east-1:891377026966:alarm:TargetTracking-endpoint/tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736/variant/AllTraffic-AlarmHigh-e394e480-4118-411f-9b4a-d836ee94d669'},
  {'AlarmName': 'TargetTracking-endpoint/tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736/variant/AllTraffic-AlarmLow-170d7ded-c4f3-422d-952c-9f38b32a9aa8',
   'AlarmARN': 'arn:aws:cloudwatch:us-east-1:891377026966:alarm:TargetTracking-endpoint/tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736/variant/AllTraffic-AlarmLow-170d7ded-c4

In [8]:
from IPython.core.display import display, HTML

display(
    HTML(
        '<b>Review <a target="blank" href="https://console.aws.amazon.com/sagemaker/home?region={}#/endpoints/{}">SageMaker REST Endpoint</a></b>'.format(
            region, tensorflow_endpoint_name
        )
    )
)

/tmp/ipykernel_5564/2023465870.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [9]:
%%time

waiter = sm.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=tensorflow_endpoint_name)

CPU times: user 23.9 ms, sys: 6.89 ms, total: 30.8 ms
Wall time: 156 ms


# Test the Deployed Model

In [10]:
import json
from sagemaker.tensorflow.model import TensorFlowPredictor
from sagemaker.serializers import JSONLinesSerializer
from sagemaker.deserializers import JSONLinesDeserializer

predictor = TensorFlowPredictor(
    endpoint_name=tensorflow_endpoint_name,
    sagemaker_session=sess,
    model_name="saved_model",
    model_version=0,
    content_type="application/jsonlines",
    accept_type="application/jsonlines",
    serializer=JSONLinesSerializer(),
    deserializer=JSONLinesDeserializer(),
)

content_type is a no-op in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


### Waiting for the Endpoint to be ready to Serve Predictions

In [11]:
import time

time.sleep(30)

# Run a Lot of Predictions and Watch the SageMaker Endpoint Scale Out

In [12]:
from IPython.core.display import display, HTML

display(
    HTML(
        '<b>Review <a target="blank" href="https://console.aws.amazon.com/sagemaker/home?region={}#/endpoints/{}">SageMaker REST Endpoint</a></b>'.format(
            region, tensorflow_endpoint_name
        )
    )
)

/tmp/ipykernel_5564/2023465870.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [13]:
inputs = [{"features": ["This is great!"]}, {"features": ["This is bad."]}]

for i in range(0, 100000):
    predicted_classes = predictor.predict(inputs)

    for predicted_class in predicted_classes:
        print("Predicted star_rating: {}".format(predicted_class))

Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted_label': 1}
Predicted star_rating: {'predicted_label': 5}
Predicted star_rating: {'predicted

KeyboardInterrupt: 

In [14]:
autoscale.describe_scaling_activities(
    ServiceNamespace="sagemaker",
    ResourceId="endpoint/" + tensorflow_endpoint_name + "/variant/AllTraffic",
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MaxResults=100
)

{'ScalingActivities': [{'ActivityId': '830d3d3c-eac9-4f18-a0e3-df0261ffc909',
   'ServiceNamespace': 'sagemaker',
   'ResourceId': 'endpoint/tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736/variant/AllTraffic',
   'ScalableDimension': 'sagemaker:variant:DesiredInstanceCount',
   'Description': 'Setting desired instance count to 2.',
   'Cause': 'monitor alarm TargetTracking-endpoint/tensorflow-training-2024-07-02-02-08-36-935-tf-1719970736/variant/AllTraffic-AlarmHigh-e394e480-4118-411f-9b4a-d836ee94d669 in state ALARM triggered policy bert-reviews-autoscale-policy',
   'StartTime': datetime.datetime(2024, 7, 3, 2, 9, 35, 378000, tzinfo=tzlocal()),
   'EndTime': datetime.datetime(2024, 7, 3, 2, 13, 26, tzinfo=tzlocal()),
   'StatusCode': 'Successful',
   'StatusMessage': 'Successfully set desired instance count to 2. Change successfully fulfilled by sagemaker.'}],
 'ResponseMetadata': {'RequestId': '2d40fa76-ba89-4280-9f10-2e70bca6204e',
  'HTTPStatusCode': 200,
  'HTTPHeaders

# Delete Endpoint
To save cost, we should delete the endpoint.

In [15]:
sm.delete_endpoint(
     EndpointName=tensorflow_endpoint_name
)

{'ResponseMetadata': {'RequestId': 'b692b9a8-c94a-429b-ae1c-90506dab3744',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'b692b9a8-c94a-429b-ae1c-90506dab3744',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Wed, 03 Jul 2024 02:15:04 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}

In [16]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [ ]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}